In [1]:
pip install groq

Note: you may need to restart the kernel to use updated packages.


In [2]:
from groq import Groq
from dotenv import load_dotenv
import os

load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
model = "llama-3.3-70b-versatile"

def add_user_message(messages, text):
    """Add YOUR message to the conversation history."""
    user_message = {
        "role": "user",    # identifies this as your message
        "content": text    # the text you typed
    }
    messages.append(user_message)


def add_assistant_message(messages, text):
    """Add the AI's reply to the conversation history."""
    assistant_message = {
        "role": "assistant",  # identifies this as AI's message
        "content": text       # the text the AI replied
    }
    messages.append(assistant_message)


def chat(messages):
    """Send the full conversation to Groq and get the AI's reply."""
    
    response = client.chat.completions.create(
        model=model,          # which AI model to use
        max_tokens=1000,      # maximum length of the reply
        messages=messages,    # the full conversation history
    )
    
    # Extract just the text from the response object
    # response.choices[0] → first (and only) reply
    # .message.content    → the actual text
    return response.choices[0].message.content


In [3]:
def chat(messages, system=None,stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop": stop_sequences
    }

    if system:
        system_message = {"role": "system", "content": system}
        params["messages"] = [system_message] + messages

    response = client.chat.completions.create(**params)

    return response.choices[0].message.content

In [6]:
messages = []

add_user_message(messages,"Generate a very short event bridge rule as json")

add_assistant_message(messages,"```json")
text=chat(messages,stop_sequences=["```"])
print(text)



{
  "event": "ORDER_COMPLETED",
  "action": "SEND_CONFIRMATION_EMAIL"
}



In [7]:
import json

json.loads(text.strip())

{'event': 'ORDER_COMPLETED', 'action': 'SEND_CONFIRMATION_EMAIL'}

In [20]:
messages = []

prompt = """
Generate three different sample AWS CLI commands. Each should be very short.
"""

add_user_message(messages, prompt)

# Prefill with "1." to force numbered list format
# and skip any preamble like "Sure! Here are..."
add_assistant_message(messages, "1.")

# Stop at "4." so it stops after exactly 3 items
text = chat(messages, stop_sequences=["4."])

# "1." was the prefill so add it back
print("1." + text.strip())



1.aws s3 ls
2. aws ec2 describe-instances
3. aws sts get-caller-identity


In [13]:
from IPython.display import Markdown
Markdown(text)

1. aws s3 ls